# 📊 Screener de Ações — Ibovespa

**Objetivo:** Identificar as 20 melhores ações do Ibovespa (excluindo bancos) que atendam a critérios fundamentalistas rigorosos e estejam com preço de mercado abaixo do preço justo calculado por DCF e Graham.

**Critérios de Screening (Ações):**
| # | Critério | Valor |
|---|----------|-------|
| 1 | P/L | entre 0 e 10 |
| 2 | P/PV | entre 0 e 1,5 |
| 3 | Margem EBIT | positiva |
| 4 | Margem Líquida | > 10% |
| 5 | Dívida Líq./EBIT | < 3 |
| 6 | Dívida Líq./PL | < 2 |
| 7 | ROE | > 10% |
| 8 | Liquidez Corrente | > 1 |
| 9 | Passivos/Ativos | < 1 |
| 10 | Liq. Média Diária | > R$ 100 mil |
| 11 | LPA | > 0 |

**Valuation:**
- DCF (Fluxo de Caixa Descontado) — taxa: Selic 14,25%, crescimento histórico
- Fórmula de Graham (com médias P/L e P/PV do setor)

**Bancos:** Lista separada com critérios adaptados.

In [4]:
# === Setup ===
import sys
import importlib
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.width', 200)

# Garantir que o diretório raiz está no path para importar src/
if '.' not in sys.path:
    sys.path.insert(0, '.')

from src import scraper, fundamentals, filters, valuation

# Recarregar módulos durante desenvolvimento
for mod in [scraper, fundamentals, filters, valuation]:
    importlib.reload(mod)

print("✅ Módulos carregados com sucesso")

✅ Módulos carregados com sucesso


## 1. Coleta de Tickers

Scraping de `dadosdemercado.com.br/acoes` para obter a lista completa de ações brasileiras, seguido de classificação em **bancos** e **não-bancos**.

In [5]:
# Obter tickers de ações brasileiras
tickers_df = scraper.get_tickers()
print(f"Total de tickers: {len(tickers_df)}")
tickers_df.head(10)

[scraper] 384 tickers obtidos de dadosdemercado.com.br
Total de tickers: 384


,ticker,ticker_sa
0,PETR4,PETR4.SA
1,B3SA3,B3SA3.SA
2,RAIZ4,RAIZ4.SA
3,ITUB4,ITUB4.SA
4,CSAN3,CSAN3.SA
5,ITSA4,ITSA4.SA
6,LREN3,LREN3.SA
7,BBDC3,BBDC3.SA
8,MGLU3,MGLU3.SA
9,COGN3,COGN3.SA


## 2. Coleta de Dados Fundamentalistas

Busca de métricas via `yfinance` para todas as ações. Isso pode levar alguns minutos dependendo do número de tickers.

In [6]:
# Coletar fundamentals para todas as ações
stock_fundamentals = fundamentals.fetch_fundamentals(tickers_df['ticker_sa'].tolist(), delay=0.4)
stock_fundamentals.head()

Coletando fundamentals: 100%|██████████| 384/384 [12:43<00:00,  1.99s/it]


[fundamentals] 384 tickers processados, 381 com dados de preço


,ticker_sa,ticker,nome,setor,industria,preco,pl,pvp,margem_ebit_pct,margem_liquida_pct,dl_ebit,dl_pl,roe_pct,liquidez_corrente,passivos_ativos,liq_media_diaria,lpa,vpa,dy_pct,divida_liquida,ebit,fcf_latest,shares_outstanding
0,PETR4.SA,PETR4,PETROBRAS PN ATZ N2,Energy,Oil & Gas Integrated,49.89,6.11,1.55,30.41,22.13,12.07,4.39,28.18,0.71,0.66,3368945977.20,8.16,32.26,7.91,333417000960.00,27614797780.70,16721236676.89,5446501379.00
1,B3SA3.SA,B3SA3,B3 ON NM,Financial Services,Financial Data & Stock Exchanges,17.05,20.54,4.92,86.09,45.54,0.13,0.06,25.59,1.91,0.64,613211775.00,0.83,3.46,3.52,1114740736.00,8666644000.00,4071357000.00,5010732147.00
2,RAIZ4.SA,RAIZ4,RAIZEN PN N2,Utilities,Utilities - Renewable,0.53,NaN,-3.44,0.22,-9.58,107.48,3.51,-231.28,1.69,0.87,31008270.10,-2.15,-0.15,NaN,61700040704.00,574045000.00,-5241328000.00,1349543297.00
3,ITUB4.SA,ITUB4,ITAUUNIBANCOPN EJ N1,Financial Services,Banks - Regional,41.45,10.34,2.23,NaN,32.28,NaN,2.86,21.01,NaN,0.93,1228924107.50,4.01,18.55,1.32,585442983936.00,NaN,27351000000.00,5403525775.00
4,CSAN3.SA,CSAN3,COSAN ON NM,Energy,Oil & Gas Refining & Marketing,5.09,NaN,3.80,0.65,-24.05,155.49,7.69,-28.96,2.58,0.77,196870612.80,-3.89,1.34,NaN,40807221248.00,262450000.00,4566917000.00,3906847275.00


## 3. Screening — Ações (não-bancos)

Aplicação dos 11 critérios fundamentalistas.

In [7]:
# Aplicar filtros fundamentalistas para ações
# Sanitizar colunas numéricas para evitar comparação str vs int nos filtros
numeric_cols = [
    'preco', 'pl', 'pvp', 'margem_ebit_pct', 'margem_liquida_pct',
    'dl_ebit', 'dl_pl', 'roe_pct', 'liquidez_corrente', 'passivos_ativos',
    'liq_media_diaria', 'lpa', 'vpa', 'dy_pct', 'divida_liquida', 'ebit',
    'fcf_latest', 'shares_outstanding'
]

stock_fundamentals_clean = stock_fundamentals.copy()
for col in numeric_cols:
    if col in stock_fundamentals_clean.columns:
        stock_fundamentals_clean[col] = pd.to_numeric(
            stock_fundamentals_clean[col], errors='coerce'
        )

filtered_stocks = filters.apply_stock_filters(stock_fundamentals_clean)

if len(filtered_stocks) > 0:
    display_cols = ['ticker', 'nome', 'setor', 'preco', 'pl', 'pvp',
                    'margem_ebit_pct', 'margem_liquida_pct', 'dl_ebit', 'dl_pl',
                    'roe_pct', 'liquidez_corrente', 'passivos_ativos', 'lpa']
    display(filtered_stocks[display_cols].style.format({
        'preco': 'R$ {:.2f}', 'pl': '{:.2f}', 'pvp': '{:.2f}',
        'margem_ebit_pct': '{:.1f}%', 'margem_liquida_pct': '{:.1f}%',
        'dl_ebit': '{:.2f}', 'dl_pl': '{:.2f}', 'roe_pct': '{:.1f}%',
        'liquidez_corrente': '{:.2f}', 'passivos_ativos': '{:.2f}', 'lpa': 'R$ {:.2f}',
    }))
else:
    print("⚠️ Nenhuma ação passou em todos os 11 critérios."
          "\nIsso pode ocorrer em mercados sobrevalorizados."
          "\nConsidere relaxar alguns critérios.")

[filters] Ações: 23/384 passaram nos 11 critérios


,ticker,nome,setor,preco,pl,pvp,margem_ebit_pct,margem_liquida_pct,dl_ebit,dl_pl,roe_pct,liquidez_corrente,passivos_ativos,lpa
0,CYRE3,CYRELA REALTON NM,Consumer Cyclical,R$ 26.09,5.71,0.93,32.2%,21.2%,1.60,0.46,22.4%,3.84,0.53,R$ 4.57
1,CMIG4,CEMIG PN EJ N1,Utilities,R$ 12.31,7.20,1.23,26.0%,11.5%,1.66,0.63,17.5%,1.00,0.54,R$ 1.71
2,RECV3,PETRORECSA ON NM,Energy,R$ 14.08,6.46,0.90,16.5%,18.8%,2.79,0.35,13.6%,2.16,0.43,R$ 2.18
3,JHSF3,JHSF PART ON NM,Real Estate,R$ 8.59,4.52,0.91,83.3%,66.0%,2.76,0.68,21.0%,2.80,0.56,R$ 1.90
4,GRND3,GRENDENE ON NM,Consumer Cyclical,R$ 4.62,6.51,1.32,28.9%,25.0%,-1.43,-0.31,17.9%,2.05,0.30,R$ 0.71
5,EZTC3,EZTEC ON NM,Real Estate,R$ 13.39,7.08,0.74,45.6%,35.7%,0.25,0.03,10.9%,8.33,0.29,R$ 1.89
6,ALUP11,ALUPAR UNT N2,Utilities,R$ 34.64,8.30,1.25,81.1%,27.6%,2.81,1.09,14.3%,2.45,0.61,R$ 4.17
7,SAPR4,SANEPAR PN N2,Utilities,R$ 8.28,6.42,0.20,40.3%,28.9%,0.62,0.14,17.9%,1.20,0.53,R$ 1.29
8,MELK3,MELNICK ON NM,Real Estate,R$ 3.35,6.32,0.65,16.0%,10.0%,2.93,0.40,11.6%,2.60,0.47,R$ 0.53
9,EVEN3,EVEN ON NM,Real Estate,R$ 7.01,4.94,0.73,9.2%,12.4%,2.59,0.29,14.0%,4.79,0.60,R$ 1.42


## 4. Valuation — Preço Justo

Cálculo do preço justo de cada ação filtrada por dois métodos:
1. **DCF (Fluxo de Caixa Descontado)** — Taxa de desconto: Selic 14,25% | Crescimento: histórico (CAGR do FCF)
2. **Fórmula de Graham** — `V = √(P/L_setor × P/PV_setor × LPA × VPA)`

Ações com preço de mercado **abaixo de ambos** os preços justos são consideradas subvalorizadas.

In [9]:
# Calcular valuation (DCF + Graham) para ações filtradas
if len(filtered_stocks) > 0:
    valued_stocks = valuation.apply_valuation(filtered_stocks, stock_fundamentals_clean)

    val_cols = ['ticker', 'nome', 'preco', 'preco_justo_dcf', 'preco_justo_graham',
                'margem_seg_dcf_pct', 'margem_seg_graham_pct', 'undervalued']
    display(valued_stocks[val_cols].style.format({
        'preco': 'R$ {:.2f}',
        'preco_justo_dcf': 'R$ {:.2f}',
        'preco_justo_graham': 'R$ {:.2f}',
        'margem_seg_dcf_pct': '{:.1f}%',
        'margem_seg_graham_pct': '{:.1f}%',
    }))
else:
    valued_stocks = pd.DataFrame()
    print("⚠️ Nenhuma ação para avaliar.")

[valuation] 16/23 ações estão abaixo do preço justo (DCF + Graham)


,ticker,nome,preco,preco_justo_dcf,preco_justo_graham,margem_seg_dcf_pct,margem_seg_graham_pct,undervalued
0,CYRE3,CYRELA REALTON NM,R$ 26.09,R$ nan,R$ 34.39,nan%,24.1%,False
1,CMIG4,CEMIG PN EJ N1,R$ 12.31,R$ 29.73,R$ 16.70,58.6%,26.3%,True
2,RECV3,PETRORECSA ON NM,R$ 14.08,R$ 63.91,R$ 19.36,78.0%,27.3%,True
3,JHSF3,JHSF PART ON NM,R$ 8.59,R$ nan,R$ 10.70,nan%,19.7%,False
4,GRND3,GRENDENE ON NM,R$ 4.62,R$ 4.80,R$ 4.80,3.8%,3.7%,True
5,EZTC3,EZTEC ON NM,R$ 13.39,R$ 14.52,R$ 14.80,7.8%,9.5%,True
6,ALUP11,ALUPAR UNT N2,R$ 34.64,R$ 50.90,R$ 43.51,31.9%,20.4%,True
7,SAPR4,SANEPAR PN N2,R$ 8.28,R$ 79.92,R$ 29.33,89.6%,71.8%,True
8,MELK3,MELNICK ON NM,R$ 3.35,R$ 4.67,R$ 4.19,28.2%,20.0%,True
9,EVEN3,EVEN ON NM,R$ 7.01,R$ nan,R$ 9.33,nan%,24.9%,False


## 5. 🏆 Top 20 Ações — Ranking Final

Ações subvalorizadas (preço de mercado < preço justo DCF **e** Graham), ordenadas por **margem de segurança média**.

In [10]:
# Filtrar apenas ações subvalorizadas e rankear
if len(valued_stocks) > 0 and valued_stocks['undervalued'].any():
    undervalued = valued_stocks[valued_stocks['undervalued']].copy()

    # Margem de segurança média entre DCF e Graham
    undervalued['margem_seg_media_pct'] = (
        undervalued[['margem_seg_dcf_pct', 'margem_seg_graham_pct']].mean(axis=1)
    )

    # Ordenar por maior margem de segurança e pegar top 20
    top20 = (undervalued
             .sort_values('margem_seg_media_pct', ascending=False)
             .head(20)
             .reset_index(drop=True))

    print(f"🏆 TOP {len(top20)} AÇÕES SUBVALORIZADAS\n")

    result_cols = ['ticker', 'nome', 'setor', 'preco', 'preco_justo_dcf',
                   'preco_justo_graham', 'margem_seg_media_pct', 'pl', 'pvp',
                   'roe_pct', 'margem_liquida_pct', 'dl_ebit']
    display(top20[result_cols].style.format({
        'preco': 'R$ {:.2f}',
        'preco_justo_dcf': 'R$ {:.2f}',
        'preco_justo_graham': 'R$ {:.2f}',
        'margem_seg_media_pct': '{:.1f}%',
        'pl': '{:.2f}',
        'pvp': '{:.2f}',
        'roe_pct': '{:.1f}%',
        'margem_liquida_pct': '{:.1f}%',
        'dl_ebit': '{:.2f}',
    }).set_caption("Top 20 — Ações subvalorizadas por margem de segurança"))
else:
    top20 = pd.DataFrame()
    print("⚠️ Nenhuma ação encontrada abaixo do preço justo (DCF + Graham).")

    if len(valued_stocks) > 0:
        # Mostrar as mais próximas de estarem subvalorizadas
        valued_stocks['margem_seg_media_pct'] = (
            valued_stocks[['margem_seg_dcf_pct', 'margem_seg_graham_pct']].mean(axis=1)
        )
        print("\nAções mais próximas do preço justo:")
        near = valued_stocks.nlargest(10, 'margem_seg_media_pct')
        display(near[['ticker', 'nome', 'preco', 'preco_justo_dcf',
                       'preco_justo_graham', 'margem_seg_media_pct']])

🏆 TOP 16 AÇÕES SUBVALORIZADAS



,ticker,nome,setor,preco,preco_justo_dcf,preco_justo_graham,margem_seg_media_pct,pl,pvp,roe_pct,margem_liquida_pct,dl_ebit
0,SAPR4,SANEPAR PN N2,Utilities,R$ 8.28,R$ 79.92,R$ 29.33,80.7%,6.42,0.20,17.9%,28.9%,0.62
1,SAPR3,SANEPAR ON N2,Utilities,R$ 9.97,R$ 159.84,R$ 29.33,79.9%,7.73,0.24,17.9%,28.9%,0.62
2,EALT4,ACO ALTONA PN,Industrials,R$ 13.26,R$ 111.05,R$ 35.07,75.1%,2.98,0.83,33.3%,18.3%,0.97
3,ALUP4,ALUPAR PN N2,Utilities,R$ 10.81,R$ 52.86,R$ 23.72,67.0%,8.72,0.39,14.3%,27.6%,2.81
4,VLID3,VALID ON NM,Industrials,R$ 20.13,R$ 83.06,R$ 35.35,59.4%,6.06,0.92,15.2%,12.7%,0.09
5,RECV3,PETRORECSA ON NM,Energy,R$ 14.08,R$ 63.91,R$ 19.36,52.6%,6.46,0.90,13.6%,18.8%,2.79
6,BLAU3,BLAU ON EJ NM,Healthcare,R$ 9.84,R$ 28.71,R$ 15.09,50.3%,7.69,1.00,13.5%,17.3%,-0.05
7,EUCA4,EUCATEX PN N1,Industrials,R$ 20.28,R$ 37.96,R$ 42.39,49.4%,5.86,0.68,12.0%,10.3%,1.62
8,CMIG4,CEMIG PN EJ N1,Utilities,R$ 12.31,R$ 29.73,R$ 16.70,42.4%,7.20,1.23,17.5%,11.5%,1.66
9,CAMB3,CAMBUCI ON EJ,Consumer Cyclical,R$ 9.55,R$ 35.01,R$ 10.51,40.9%,5.86,1.31,23.3%,17.9%,-0.87


## 7. Resumo da Análise

In [11]:
# Sumário
print("=" * 70)
print("RESUMO DA ANÁLISE")
print("=" * 70)
print(f"Total de tickers analisados:     {len(tickers_df)}")
print(f"\nAções que passaram nos filtros:   {len(filtered_stocks)}")
if len(valued_stocks) > 0:
    print(f"Ações subvalorizadas:            {valued_stocks['undervalued'].sum()}")
print(f"Top 20 selecionadas:             {len(top20)}")
print("=" * 70)
print(f"\nParâmetros DCF: Selic={valuation.SELIC*100:.2f}%, "
      f"Crescimento terminal={valuation.TERMINAL_GROWTH*100:.1f}%, "
      f"Projeção={valuation.PROJECTION_YEARS} anos")
print("Graham: V = √(P/L_setor × P/PV_setor × LPA × VPA)")

RESUMO DA ANÁLISE
Total de tickers analisados:     384

Ações que passaram nos filtros:   23
Ações subvalorizadas:            16
Top 20 selecionadas:             16

Parâmetros DCF: Selic=14.25%, Crescimento terminal=3.5%, Projeção=5 anos
Graham: V = √(P/L_setor × P/PV_setor × LPA × VPA)
